# torch.package — Ship Models with Their Code

**What you'll learn:**
- Why `torch.save` isn't enough for model deployment
- `PackageExporter`: packaging models with their dependencies
- `intern`, `extern`, `mock`, `deny` — dependency control
- `PackageImporter`: loading packages in isolated environments
- Inspecting package contents
- Comparison with `torch.save` and `torch.export`

**Prerequisites:** Basic `nn.Module` and `torch.save`/`torch.load` usage.

**All code runs on CPU.** Packages are saved to a temp directory.

In [ ]:
import torch
import torch.nn as nn
from torch.package import PackageExporter, PackageImporter
import tempfile
import os

# Create a temp directory for our packages
pkg_dir = tempfile.mkdtemp(prefix="torch_package_demo_")
print(f"PyTorch version: {torch.__version__}")
print(f"Package output directory: {pkg_dir}")

## The Problem: "Works on My Machine"

`torch.save(model.state_dict(), "model.pt")` only saves **weights**. To load:
- You need the exact model class definition
- You need all the same imports available
- If you refactor the code, old checkpoints break

`torch.save(model, "model.pt")` (pickle) saves the model object, but:
- It stores *references* to code, not the code itself
- Moving the file to a different machine often fails
- Pickle is fragile across Python/PyTorch versions

**`torch.package` solves this** by bundling model weights + source code + dependencies into a single portable archive.

## Define a Simple Model

Let's create a model to package. In real life, this might live in `my_project/models/classifier.py`.

In [ ]:
class SimpleClassifier(nn.Module):
    """A small classifier for demonstration."""
    def __init__(self, input_dim=32, hidden_dim=64, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )
        self.input_dim = input_dim
        self.num_classes = num_classes

    def forward(self, x):
        return self.net(x)

model = SimpleClassifier()
model.eval()

# Create a test input and reference output
test_input = torch.randn(4, 32)
with torch.no_grad():
    reference_output = model(test_input)

print(f"Model: SimpleClassifier")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Input: {test_input.shape} → Output: {reference_output.shape}")

## PackageExporter Basics

The `PackageExporter` creates a `.pt` archive. You control what goes inside with:

| Method | Effect |
|--------|--------|
| `intern(pattern)` | Bundle this module's source code INTO the package |
| `extern(pattern)` | Mark as external dependency (must exist at load time) |
| `mock(pattern)` | Replace with a stub (for unused imports) |
| `deny(pattern)` | Error if this module is needed |

In [ ]:
pkg_path = os.path.join(pkg_dir, "classifier.pt")

with PackageExporter(pkg_path) as exporter:
    # torch and its submodules are external (user must have PyTorch installed)
    exporter.extern("torch.**")
    exporter.extern("numpy.**")

    # Intern __main__ — this bundles our model class definition
    exporter.intern("__main__")

    # Save the model object
    exporter.save_pickle("model", "classifier.pkl", model)

print(f"Package saved: {pkg_path}")
print(f"File size: {os.path.getsize(pkg_path) / 1024:.1f} KB")

## Loading with PackageImporter

`PackageImporter` loads the package and gives you access to everything inside. The model class is loaded from the package itself — no need for the original source.

In [ ]:
importer = PackageImporter(pkg_path)
loaded_model = importer.load_pickle("model", "classifier.pkl")

# Verify the loaded model produces the same output
loaded_model.eval()
with torch.no_grad():
    loaded_output = loaded_model(test_input)

print(f"Loaded model type: {type(loaded_model).__name__}")
print(f"Outputs match: {torch.allclose(reference_output, loaded_output)}")
print(f"Max difference: {(reference_output - loaded_output).abs().max().item():.2e}")

## Saving Multiple Objects

A package can contain multiple pickled objects — model, config, metadata, preprocessing info — all in one file.

In [ ]:
multi_pkg_path = os.path.join(pkg_dir, "full_deployment.pt")

config = {
    "model_version": "1.2.0",
    "input_dim": 32,
    "num_classes": 10,
    "preprocessing": "normalize_mean_std",
    "class_names": [f"class_{i}" for i in range(10)],
}

with PackageExporter(multi_pkg_path) as exporter:
    exporter.extern("torch.**")
    exporter.intern("__main__")

    # Save multiple objects under different names
    exporter.save_pickle("model", "classifier.pkl", model)
    exporter.save_pickle("config", "config.pkl", config)

    # Save a text file (e.g., release notes)
    exporter.save_text("metadata", "version.txt", "v1.2.0 - added dropout")

print(f"Multi-object package: {os.path.getsize(multi_pkg_path) / 1024:.1f} KB")

# Load everything back
imp = PackageImporter(multi_pkg_path)
loaded_cfg = imp.load_pickle("config", "config.pkl")
loaded_txt = imp.load_text("metadata", "version.txt")

print(f"\nLoaded config: {loaded_cfg}")
print(f"Loaded text: {loaded_txt}")

## Inspecting Package Contents

Packages are ZIP files internally. You can list their contents to see what got bundled.

In [ ]:
import zipfile

# Method 1: Use file_structure() from the importer
imp = PackageImporter(multi_pkg_path)
print("Package file structure:")
print(imp.file_structure())

# Method 2: Peek inside the zip directly
print("\n\nRaw ZIP contents:")
with zipfile.ZipFile(multi_pkg_path, "r") as zf:
    for info in zf.infolist():
        print(f"  {info.filename:50s} {info.file_size:>8d} bytes")

## Mock — Replace Unused Dependencies

If your model file imports a library that's only used at training time (e.g., `matplotlib` for visualization), you can `mock` it so the package doesn't need it at load time.

In [ ]:
# Model that imports something only for training/viz
class ModelWithVizImport(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(16, 4)

    def forward(self, x):
        return self.linear(x)

    def plot_weights(self):
        import matplotlib.pyplot as plt  # only used here
        plt.hist(self.linear.weight.detach().numpy().flatten())

mock_pkg_path = os.path.join(pkg_dir, "model_mocked.pt")

with PackageExporter(mock_pkg_path) as exporter:
    exporter.extern("torch.**")
    exporter.intern("__main__")
    # Mock matplotlib — it won't be needed at inference time
    exporter.mock("matplotlib.**")

    exporter.save_pickle("model", "model.pkl", ModelWithVizImport())

# Load — works even without matplotlib!
imp = PackageImporter(mock_pkg_path)
loaded = imp.load_pickle("model", "model.pkl")
result = loaded(torch.randn(2, 16))
print(f"Model loaded successfully (matplotlib mocked out)")
print(f"Inference works: {result.shape}")

## Intern vs Extern: What Gets Bundled?

Understanding the difference:
- **intern**: Source code is copied INTO the package. The package is self-contained for this module.
- **extern**: The module is NOT included. The loading environment must have it installed.

**Rule of thumb:**
- `intern` your own code (model definitions, custom layers)
- `extern` well-known libraries (torch, numpy, PIL)
- `mock` training-only dependencies (matplotlib, tensorboard, wandb)

In [ ]:
# Demonstrate: read back the interned source code
imp = PackageImporter(multi_pkg_path)

# The package contains the source for __main__
print("Interned source code inside the package:")
print("=" * 50)
try:
    src = imp.zip_reader.get_record(".data/python_files/__main__.py")
    # Show first few lines
    lines = src.decode('utf-8') if isinstance(src, bytes) else str(src)
    for line in lines.split('\n')[:15]:
        print(f"  {line}")
    print("  ...")
except Exception as e:
    # Alternative: check what files are interned
    print(f"  (Source bundled in package — visible in file_structure above)")
    fs = str(imp.file_structure())
    for line in fs.split('\n'):
        if '.py' in line:
            print(f"  {line}")

## Deny — Prevent Accidental Dependencies

Use `deny` to make packaging FAIL if an unwanted module gets pulled in. This is a safety check — e.g., prevent accidentally bundling sensitive internal code.

In [ ]:
deny_pkg_path = os.path.join(pkg_dir, "denied.pt")

# This will work because SimpleClassifier doesn't use 'secrets'
try:
    with PackageExporter(deny_pkg_path) as exporter:
        exporter.extern("torch.**")
        exporter.intern("__main__")
        exporter.deny("secrets")  # block the 'secrets' module
        exporter.save_pickle("model", "model.pkl", model)
    print("Package created successfully (denied module wasn't needed)")
except Exception as e:
    print(f"Packaging failed: {e}")

# If the model DID try to use 'secrets', packaging would fail
print("\n'deny' acts as a guardrail — packaging fails if the denied module is needed")

## Packaging a More Complex Model

Let's package a model with custom components to see how `intern` handles multiple classes.

In [ ]:
class Attention(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.scale = (dim // heads) ** -0.5

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.heads, C // self.heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)

class TransformerBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = Attention(dim)
        self.ffn = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class MiniTransformer(nn.Module):
    def __init__(self, dim=64, depth=4, num_classes=10):
        super().__init__()
        self.embed = nn.Linear(32, dim)
        self.blocks = nn.Sequential(*[TransformerBlock(dim) for _ in range(depth)])
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x):
        x = self.embed(x).unsqueeze(1)  # add seq dim
        x = self.blocks(x)
        return self.head(x.squeeze(1))

transformer = MiniTransformer()
transformer.eval()
test_x = torch.randn(2, 32)
with torch.no_grad():
    ref_out = transformer(test_x)
print(f"MiniTransformer: {sum(p.numel() for p in transformer.parameters()):,} params")
print(f"Output: {ref_out.shape}")

In [ ]:
transformer_pkg_path = os.path.join(pkg_dir, "transformer.pt")

with PackageExporter(transformer_pkg_path) as exporter:
    exporter.extern("torch.**")
    exporter.intern("__main__")  # bundles Attention, TransformerBlock, MiniTransformer
    exporter.save_pickle("model", "transformer.pkl", transformer)
    exporter.save_pickle("meta", "info.pkl", {
        "dim": 64, "depth": 4, "num_classes": 10, "architecture": "MiniTransformer"
    })

# Verify round-trip
imp = PackageImporter(transformer_pkg_path)
loaded_transformer = imp.load_pickle("model", "transformer.pkl")
loaded_meta = imp.load_pickle("meta", "info.pkl")

loaded_transformer.eval()
with torch.no_grad():
    loaded_out = loaded_transformer(test_x)

print(f"Loaded architecture: {loaded_meta['architecture']}")
print(f"Outputs match: {torch.allclose(ref_out, loaded_out, atol=1e-6)}")
print(f"Package size: {os.path.getsize(transformer_pkg_path) / 1024:.1f} KB")

## Comparison: torch.save vs torch.package vs torch.export

| Feature | `torch.save` | `torch.package` | `torch.export` |
|---------|-------------|-----------------|----------------|
| Saves weights | Yes | Yes | Yes |
| Saves code | No (pickle refs) | Yes (source) | Yes (IR graph) |
| Portable across machines | Fragile | Yes | Yes |
| Requires original class | Yes (state_dict) | No | No |
| Python code bundled | No | Yes | No (graph only) |
| Supports dynamic control flow | N/A | Yes (Python) | Limited |
| Production serving | Not ideal | Good | Best |
| Package size | Small | Medium | Medium |
| Execution mode | Python | Python | Graph-based |

**When to use each:**
- `torch.save` — Quick checkpoints during training (same codebase)
- `torch.package` — Ship a model to a different team/machine with full Python flexibility
- `torch.export` — Production deployment with maximum optimization potential

## Hermetic Imports: Package Isolation

A loaded package creates an isolated namespace. Two different packages can have different versions of the same module without conflicts.

In [ ]:
# Load the same package twice — they are independent namespaces
imp1 = PackageImporter(transformer_pkg_path)
imp2 = PackageImporter(transformer_pkg_path)

model1 = imp1.load_pickle("model", "transformer.pkl")
model2 = imp2.load_pickle("model", "transformer.pkl")

# They are the same model but loaded independently
print(f"model1 class: {type(model1)}")
print(f"model2 class: {type(model2)}")
print(f"Same class object? {type(model1) is type(model2)}")
print(f"\nEach import is hermetic — classes are loaded fresh from the package")

## Try It Yourself

**Exercise:** Package a model with its config and load it in a fresh importer.

1. Define a simple model (e.g., a 2-layer MLP)
2. Create a config dict with hyperparameters
3. Package both using `PackageExporter` (intern `__main__`, extern `torch.**`)
4. Load with `PackageImporter` and verify the model output matches

In [ ]:
# YOUR CODE HERE
# Step 1: Define your model
# class MyMLP(nn.Module): ...

# Step 2: Create config
# my_config = {"lr": 0.001, "epochs": 10, ...}

# Step 3: Package it
# exercise_path = os.path.join(pkg_dir, "exercise.pt")
# with PackageExporter(exercise_path) as exp:
#     exp.extern("torch.**")
#     exp.intern("__main__")
#     exp.save_pickle("model", "model.pkl", my_model)
#     exp.save_pickle("config", "config.pkl", my_config)

# Step 4: Load and verify
# imp = PackageImporter(exercise_path)
# loaded = imp.load_pickle("model", "model.pkl")
# ...

print("Complete the exercise above!")

## Key Takeaways

1. **`torch.package`** bundles model weights + source code + metadata into a single portable `.pt` archive
2. **`intern(pattern)`** — copies module source INTO the package (use for your own code)
3. **`extern(pattern)`** — marks a dependency as external (must be installed at load time)
4. **`mock(pattern)`** — replaces a module with a stub (for unused training-time deps)
5. **`deny(pattern)`** — fails packaging if the module is needed (safety guardrail)
6. **`save_pickle` / `save_text`** — save any Python object or text file into the package
7. **`PackageImporter`** creates a hermetic namespace — no class definition conflicts
8. **Multiple objects** can live in one package (model + config + preprocessing + metadata)
9. **Use `torch.package`** when shipping models to other teams/machines that may not have your source tree
10. **Use `torch.export`** instead when you need maximum inference optimization or non-Python runtimes